# CoXAM Multi-Dataset (Between-Subjects) Workflow

Checks whether a `dataset` between-subjects factor with levels
`["wine_quality", "mushrooms"]` is handled correctly at each stage:
**study design -> generate trials -> simulation**, kept as three distinct
steps below (per arm). Every stage calls the real `xaikitTest` API function
directly -- no wrapper functions -- looped once per dataset level.

**Architecture note, confirmed while building this.** `xaikitTest` prepares
exactly one dataset at a time (`self.data`, `self.model`, `self.combined_explanations`
are all single-dataset state) -- so `generate_trials` cannot natively serve two
different datasets' instances/features from one study object. Adding
`dataset` as an ordinary IV on a single study would only *label* trials with
two dataset names while every trial's actual feature values still came from
whichever one dataset happened to be prepared.

The correct construction, used below: build **one study object per dataset
level**, each going through the normal single-dataset pipeline, assign each
arm a disjoint block of participant ids so the between-subjects factor is
real at the participant level, tag each arm's rows with an explicit
`dataset` column, then concatenate. Both datasets here are corpus-covered
(see `COXAM_CORPUS_FEATURES`), so both arms simulate with `source="assets"`
-- no AI training, no explanation fitting.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "tutorials" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import src as xk

from src.virtual_experiment_executor.experiment_simualtion.CoXAM.coxam_trial_executor import (
    COXAM_CORPUS_FEATURES,
    coxam_available_instance_ids,
)

OUTPUT_DIR = REPO_ROOT / "tutorials" / "coxam_multidataset_workflow_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

XAI_TYPES = ["decision_tree", "logistic_regression"]
TESTED_W_XAI = [True, False]
DATASET_LEVELS = ["wine_quality", "mushrooms"]
PARTICIPANTS_PER_CONDITION = 4

# Disjoint participant-id blocks per arm -- what makes `dataset` a genuine
# between-subjects factor once the two arms are combined below.
PARTICIPANT_OFFSETS = {"wine_quality": 0, "mushrooms": 1000}

studies: dict[str, "xk.xaikitTest"] = {}
trials_by_dataset: dict[str, pd.DataFrame] = {}
simulated_by_dataset: dict[str, pd.DataFrame] = {}

print("corpus covers:", list(COXAM_CORPUS_FEATURES.keys()))


## 2. Study Design (per arm)

Each arm declares the *same* within-subject IVs, CVs and DV, plus `dataset`
as a between-subjects IV -- for `validate_design`/documentation. The actual
between-subjects split is enforced later by which arm a participant belongs
to. `prepare_dataset(..., cognitive_model_id="coxam")` routes each arm onto
the corpus's own 6-feature set for that dataset.

In [ ]:
for app_id in DATASET_LEVELS:
    study = xk.xaikitTest(output_dir=OUTPUT_DIR / app_id)
    study.add_iv("xai_type", "within", XAI_TYPES, randomization="block")
    study.add_iv("tested_w_xai", "within", TESTED_W_XAI, randomization="trial")
    study.add_iv("dataset", "between", DATASET_LEVELS, randomization=None)
    study.add_cv("user_task", ["forward_simulation"])
    study.add_dv("forward_accuracy", ["continuous"])
    study.validate_design(show=False)

    study.prepare_dataset(
        dataset_id=app_id,
        model_type="mlp",
        cognitive_model_id="coxam",
        show_available=False,
        show_summary=False,
    )
    studies[app_id] = study
    print(app_id, "features:", study.data.raw_feature_names)


## 3. Generate Trials (per arm)

`allowed_instance_ids` keeps every trial to an instance the published corpus
can serve. `study.trials` is a list of row dicts, not a DataFrame --
`_require_trials()` checks it with a plain `if not self.trials`, which
breaks if a DataFrame is assigned there instead, so the offset/tag edits
below convert back with `.to_dict(orient="records")` before writing onto the
study.

In [ ]:
for app_id in DATASET_LEVELS:
    study = studies[app_id]
    instance_ids = coxam_available_instance_ids(app_id)

    study.generate_trials(
        model_name="mlp",
        participants_per_between_condition=PARTICIPANTS_PER_CONDITION,
        allowed_instance_ids=instance_ids,
        counterbalancing_strategy="balanced_latin_square",
        trial_randomization_strategy="balanced",
        output_dir="trials",
        show=False,
    )

    trials = pd.DataFrame(study.trials).copy()
    trials["participantId"] = trials["participantId"] + PARTICIPANT_OFFSETS[app_id]
    trials["dataset"] = app_id
    study.trials = trials.to_dict(orient="records")
    study.trial_result.trials = trials.to_dict(orient="records")
    trials_by_dataset[app_id] = trials

    print(app_id, "participants:", sorted(trials["participantId"].unique()))

combined_trials = pd.concat(trials_by_dataset.values(), ignore_index=True)

overlap = set(trials_by_dataset["wine_quality"]["participantId"]) & set(trials_by_dataset["mushrooms"]["participantId"])
assert not overlap, f"between-subjects violated: participant ids overlap: {overlap}"
per_participant_levels = combined_trials.groupby("participantId")["dataset"].nunique()
assert (per_participant_levels == 1).all(), "a participant saw more than one dataset level"

print("participant id overlap:", overlap)
print("combined trials shape :", combined_trials.shape)
print(combined_trials["dataset"].value_counts())


## 4. Simulation (per arm)

`source="assets"` reads each dataset's published surrogates instead of
fitting new ones, which is what lets both arms run with no trained model.
Passing `source="fit"` here would raise, since fitting needs a trained model
neither arm has. `run_experiment` is called directly -- it's `xaikitTest`'s
own dispatcher (`coxam` -> `run_coxam_study`), not something this notebook
wraps.

In [ ]:
for app_id in DATASET_LEVELS:
    study = studies[app_id]
    study.set_cognitive_model(cognitive_model_id="coxam")

    simulated = study.run_experiment(mode="whole_experiment", source="assets").copy()
    simulated["participantId"] = simulated["participantId"] + PARTICIPANT_OFFSETS[app_id]
    simulated["dataset"] = app_id
    simulated_by_dataset[app_id] = simulated

    print(app_id, "no AI model trained:", study.trained_ai_model is None, "| simulated rows:", len(simulated))

combined_sim = pd.concat(simulated_by_dataset.values(), ignore_index=True)
print()
print("combined simulated-results shape:", combined_sim.shape)


## 5. Analysis: forward_accuracy by dataset

In [ ]:
from src.statistical_analyst import analyze_iv_dv, pairwise_condition_tests
from src.result_visualizer import plot_iv_dv_grid

print(combined_sim.groupby(["dataset", "xai_type", "tested_w_xai"])["forward_accuracy"]
      .agg(["mean", "size"]).round(3))


In [ ]:
result = analyze_iv_dv(combined_sim, iv="dataset", dv="forward_accuracy")
print("method:", result.method)
print(result.descriptives)


In [ ]:
posthoc = pairwise_condition_tests(
    combined_sim, value_col="forward_accuracy", condition_cols=["dataset"],
)
print(posthoc.table)


In [ ]:
grid = plot_iv_dv_grid(
    combined_sim,
    ivs=["dataset", "xai_type", "tested_w_xai"],
    dvs=["forward_accuracy"],
    phase=None,
    title="CoXAM: wine_quality vs mushrooms (between-subjects)",
)


## 6. What This Notebook Shows

- A `dataset` between-subjects IV with levels `wine_quality`/`mushrooms` runs
  correctly through three distinct stages -- **design** (Section 2),
  **generate trials** (Section 3), **simulate** (Section 4) -- when built as
  two study arms with disjoint participant blocks, rather than as an IV on
  one shared study object. Confirmed no participant id overlap and every
  participant maps to exactly one level.
- Every stage calls the real `xaikitTest` API method directly inside a loop
  over `DATASET_LEVELS` -- `prepare_dataset`, `generate_trials`,
  `run_experiment` -- with no wrapper functions hiding them.
- `source="assets"` simulation (no AI training, no explanation fitting) and
  analysis (`analyze_iv_dv`, `pairwise_condition_tests`, `plot_iv_dv_grid`)
  all work unmodified on the combined table once `dataset` is a real column.
- This composes existing single-dataset primitives; it does not change
  `generate_trials`/`prepare_dataset` to natively understand a `dataset` IV
  on one study object. If the design UI needs to export a `dataset` IV
  directly, the design-export/server layer would need to detect it and
  perform this same per-level study composition (design, then trials, then
  simulation, once per level) automatically.